# Probability, conditional probability and Bayes' rule

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## Why probability?
Data is noisy, labels are sometimes wrong and the future is not certain. A good model does not say "this email is spam"; it says "this is spam **with probability 97%**". Probability is the language for talking about uncertainty.

## The basics with one die
| Concept | Definition | Example with a die |
|---|---|---|
| Sample space $\Omega$ | All possible outcomes | $\{1,2,3,4,5,6\}$ |
| Event | A subset of the sample space | A = "even" = $\{2,4,6\}$ |
| Probability | A number between 0 and 1 | $P(A)=3/6=0.5$ |
| Complement | A does not happen | $P(A^c)=1-0.5=0.5$ |

## The rules of probability
A = "even" = $\{2,4,6\}$ and B = "greater than 4" = $\{5,6\}$:

| Rule | Formula | Calculation |
|---|---|---|
| Complement | $P(A^c)=1-P(A)$ | $1-\frac36=\frac12$ |
| Intersection (and) | $P(A\cap B)$ | Only $\{6\}$: $\frac16$ |
| Union (or) | $P(A\cup B)=P(A)+P(B)-P(A\cap B)$ | $\frac36+\frac26-\frac16=\frac46$ |
| Conditional | $P(A\mid B)=\dfrac{P(A\cap B)}{P(B)}$ | $\dfrac{1/6}{2/6}=\frac12$ |
| Product | $P(A\cap B)=P(A\mid B)\,P(B)$ | $\frac12\times\frac26=\frac16$ |

## Conditional probability
$P(A\mid B)$ is "the probability of A **when we know** B happened". Knowing B shrinks the sample space: if we know the roll was greater than 4, only $\{5,6\}$ is possible, and one of those two is even.

## Independence
Two events are **independent** if knowing one says nothing about the other:

$$P(A\cap B)=P(A)\,P(B)\quad\Longleftrightarrow\quad P(A\mid B)=P(A)$$

In the example above $P(A)P(B)=\frac12\times\frac13=\frac16=P(A\cap B)$, so "even" and "greater than 4" are independent. Two coin flips are independent too: $P(\text{HH})=0.5\times0.5=0.25$.

## Bayes' rule
The most important formula of this lesson. We often know $P(\text{evidence}\mid\text{cause})$ but want $P(\text{cause}\mid\text{evidence})$:

$$P(H\mid E)=\frac{P(E\mid H)\,P(H)}{P(E)}$$

| Part | Name | Meaning |
|---|---|---|
| $P(H)$ | Prior | What we believe before seeing the evidence |
| $P(E\mid H)$ | Likelihood | How likely the evidence is if the hypothesis is true |
| $P(E)$ | Evidence | The overall chance of seeing the evidence |
| $P(H\mid E)$ | Posterior | The updated belief |

## Example: a medical test
A disease affects 1% of people. A test detects 90% of sick people (sensitivity) and also wrongly comes out positive for 10% of healthy people. Your test is positive. How likely is it that you are sick?

Most people say 90%. Let us count with 1000 people:

> [Of the 108 people who test positive, only 9 are really sick.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_bayes_tree.png)

*Of the 108 people who test positive, only 9 are really sick.*

| Step | Calculation | Value |
|---|---|---|
| $P(+\mid D)P(D)$ | $0.9\times0.01$ | 0.009 |
| $P(+\mid\text{healthy})P(\text{healthy})$ | $0.1\times0.99$ | 0.099 |
| $P(+)$ | $0.009+0.099$ | 0.108 |
| $P(D\mid+)$ | $0.009/0.108$ | 0.083 |

Only **8.3%**! Because the disease is rare, false positives far outnumber true positives. This common mistake is called "base-rate neglect".

> Open the interactive lab for this part on the website: https://behrooztaheri.github.io/learn-python-ai/#/l/math-prob

> In the lab raise the prevalence from 1% to 20%: the same test is now far more trustworthy. Machine learning behaves the same way: when the positive class is rare, precision drops even for an "accurate" model.

## Bayes in machine learning
| Where | How |
|---|---|
| Naive Bayes | $P(\text{class}\mid\text{features})\propto P(\text{class})\prod_i P(x_i\mid\text{class})$ |
| Spam filters | The spam probability is updated with every word seen |
| Regularization | A prior on the weights = an L2 or L1 penalty |
| Generative models | $P(x\mid z)P(z)$ |

Spam example: 40% of emails are spam. The word "free" appears in 50% of spam and 5% of normal emails. If an email contains "free":

$$P(\text{spam}\mid\text{free})=\frac{0.5\times0.4}{0.5\times0.4+0.05\times0.6}=\frac{0.20}{0.23}\approx0.87$$



In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# Rules of probability, checked by rolling a die one million times
rolls = rng.integers(1, 7, 1_000_000)
A = rolls % 2 == 0          # even
B = rolls > 4               # greater than 4
print("P(A)        =", A.mean().round(3), "  (exact 0.5)")
print("P(A and B)  =", (A & B).mean().round(3), "  (exact 0.167)")
print("P(A or B)   =", (A | B).mean().round(3), "  (exact 0.667)")
print("P(A | B)    =", A[B].mean().round(3), "  (exact 0.5)")

def bayes(prior, sensitivity, false_positive):
    evidence = sensitivity * prior + false_positive * (1 - prior)
    return sensitivity * prior / evidence

print("P(sick | +) with 1% prevalence :", round(bayes(0.01, 0.9, 0.1), 4))
print("P(sick | +) with 20% prevalence:", round(bayes(0.20, 0.9, 0.1), 4))
print("P(spam | 'free')               :", round(bayes(0.4, 0.5, 0.05), 4))

# Simulation of the medical test with 1,000,000 people
sick = rng.random(1_000_000) < 0.01
positive = np.where(sick, rng.random(1_000_000) < 0.9, rng.random(1_000_000) < 0.1)
print("simulated P(sick | +):", sick[positive].mean().round(4))

# A second positive test: yesterday's posterior becomes today's prior
p = 0.01
for test in range(1, 4):
    p = bayes(p, 0.9, 0.1)
    print(f"after {test} positive test(s): {p:.3f}")

### Exercise

In a factory machine A makes 60% of the parts and machine B 40%. 2% of A's parts and 5% of B's are defective. If a part is defective, what is the probability it came from B?

In [ ]:
# Your answer here

### Exercise

In the lab set sensitivity and specificity to 99% and prevalence to 0.5%. Does a positive test mean you are sick?

In [ ]:
# Your answer here